# Tema 5 — Hoja de Ejercicios III
# Tokenizadores en modelos Transformer

Este notebook resuelve en un único archivo todos los ejercicios de la **Hoja de Ejercicios III**.

El objetivo es analizar el comportamiento de distintos tokenizadores empleados por modelos Transformer:

- WordPiece con `bert-base-uncased`.
- Tokens especiales: `[CLS]`, `[SEP]`, `[PAD]`.
- Máscaras de atención.
- Padding y truncado.
- Comparación entre un tokenizador generalista y uno especializado en dominio biomédico.

No vamos a entrenar modelos; nos centraremos en cómo se preparan los textos antes de introducirlos en un Transformer.


## 0. Instalación de librerías

En Google Colab, si no tienes `transformers` instalado, ejecuta:

```python
!pip install transformers -q
```

Si trabajas en local con `uv`:

```bash
uv add transformers pandas
```


In [ ]:
# En Colab, descomenta si hace falta:
# !pip install -q transformers pandas

## 1. Importación de librerías

Usaremos:

- `AutoTokenizer`: para cargar tokenizadores de Hugging Face.
- `pandas`: para mostrar tablas comparativas.


In [1]:
from transformers import AutoTokenizer
import pandas as pd

# Ejercicio 1. Preparación de textos de entrada

El Ejercicio 1 se divide en tres apartados:

- **Apartado a)** Tokenización con WordPiece.
- **Apartado b)** Tokens especiales y máscara de atención.
- **Apartado c)** Padding, truncado y longitud máxima.

Para estos apartados usaremos el tokenizador de BERT:

```python
bert-base-uncased
```

Este modelo trabaja en inglés y no distingue mayúsculas/minúsculas.


## Ejercicio 1.a — Tokenización con WordPiece

WordPiece divide palabras poco frecuentes o desconocidas en subtokens.

En BERT, los subtokens que continúan una palabra anterior se marcan con el prefijo:

```text
##
```

Por ejemplo, si una palabra no aparece completa en el vocabulario, puede dividirse en varias piezas.


In [2]:
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

words = [
    "unhappiness",
    "tokenization",
    "pretraining",
    "unbelievable",
    "internationalization",
    "hyperparametrization",
    "distillation",
    "contextualization",
]

print("Tokenizador cargado:", bert_tokenizer.name_or_path)

Tokenizador cargado: bert-base-uncased


### 1.a.1 Tokenizar cada palabra

Usamos:

```python
tokenizer.tokenize(word)
```

Esto devuelve los subtokens generados por WordPiece sin añadir tokens especiales.


In [3]:
for word in words:
    tokens = bert_tokenizer.tokenize(word)
    print(f"{word:25s} -> {tokens}")

unhappiness               -> ['un', '##ha', '##pp', '##iness']
tokenization              -> ['token', '##ization']
pretraining               -> ['pre', '##train', '##ing']
unbelievable              -> ['unbelievable']
internationalization      -> ['international', '##ization']
hyperparametrization      -> ['hyper', '##para', '##met', '##rization']
distillation              -> ['di', '##sti', '##llation']
contextualization         -> ['context', '##ual', '##ization']


### 1.a.2 Obtener ids con `tokenizer.encode`

Usamos:

```python
tokenizer.encode(word)
```

Esto convierte la palabra en identificadores numéricos.  
Por defecto se añaden tokens especiales `[CLS]` y `[SEP]`.


In [4]:
encoded_words = {}

for word in words:
    ids = bert_tokenizer.encode(word)
    encoded_words[word] = ids
    tokens = bert_tokenizer.convert_ids_to_tokens(ids)

    print(f"Palabra: {word}")
    print("IDs:", ids)
    print("Tokens:", tokens)
    print("-" * 80)

Palabra: unhappiness
IDs: [101, 4895, 3270, 9397, 9961, 102]
Tokens: ['[CLS]', 'un', '##ha', '##pp', '##iness', '[SEP]']
--------------------------------------------------------------------------------
Palabra: tokenization
IDs: [101, 19204, 3989, 102]
Tokens: ['[CLS]', 'token', '##ization', '[SEP]']
--------------------------------------------------------------------------------
Palabra: pretraining
IDs: [101, 3653, 23654, 2075, 102]
Tokens: ['[CLS]', 'pre', '##train', '##ing', '[SEP]']
--------------------------------------------------------------------------------
Palabra: unbelievable
IDs: [101, 23653, 102]
Tokens: ['[CLS]', 'unbelievable', '[SEP]']
--------------------------------------------------------------------------------
Palabra: internationalization
IDs: [101, 2248, 3989, 102]
Tokens: ['[CLS]', 'international', '##ization', '[SEP]']
--------------------------------------------------------------------------------
Palabra: hyperparametrization
IDs: [101, 23760, 28689, 11368,

### 1.a.3 Reconstruir palabras con `tokenizer.decode`

Usamos:

```python
tokenizer.decode(ids)
```

Mostramos dos versiones:

- Con tokens especiales.
- Sin tokens especiales.


In [5]:
for word, ids in encoded_words.items():
    decoded_with_specials = bert_tokenizer.decode(ids)
    decoded_without_specials = bert_tokenizer.decode(ids, skip_special_tokens=True)

    print(f"Palabra original: {word}")
    print("Decode con tokens especiales:", decoded_with_specials)
    print("Decode sin tokens especiales:", decoded_without_specials)
    print("-" * 80)

Palabra original: unhappiness
Decode con tokens especiales: [CLS] unhappiness [SEP]
Decode sin tokens especiales: unhappiness
--------------------------------------------------------------------------------
Palabra original: tokenization
Decode con tokens especiales: [CLS] tokenization [SEP]
Decode sin tokens especiales: tokenization
--------------------------------------------------------------------------------
Palabra original: pretraining
Decode con tokens especiales: [CLS] pretraining [SEP]
Decode sin tokens especiales: pretraining
--------------------------------------------------------------------------------
Palabra original: unbelievable
Decode con tokens especiales: [CLS] unbelievable [SEP]
Decode sin tokens especiales: unbelievable
--------------------------------------------------------------------------------
Palabra original: internationalization
Decode con tokens especiales: [CLS] internationalization [SEP]
Decode sin tokens especiales: internationalization
-------------

### 1.a.4 Tabla resumen

Creamos una tabla para comparar palabra, subtokens, ids y reconstrucción.


In [6]:
rows = []

for word in words:
    tokens = bert_tokenizer.tokenize(word)
    ids = bert_tokenizer.encode(word)
    decoded = bert_tokenizer.decode(ids, skip_special_tokens=True)

    rows.append({
        "Palabra": word,
        "Subtokens WordPiece": tokens,
        "IDs": ids,
        "Decode sin especiales": decoded
    })

df_wordpiece = pd.DataFrame(rows)
df_wordpiece

,Palabra,Subtokens WordPiece,IDs,Decode sin especiales
0,unhappiness,"[un, ##ha, ##pp, ##iness]","[101, 4895, 3270, 9397, 9961, 102]",unhappiness
1,tokenization,"[token, ##ization]","[101, 19204, 3989, 102]",tokenization
2,pretraining,"[pre, ##train, ##ing]","[101, 3653, 23654, 2075, 102]",pretraining
3,unbelievable,[unbelievable],"[101, 23653, 102]",unbelievable
4,internationalization,"[international, ##ization]","[101, 2248, 3989, 102]",internationalization
5,hyperparametrization,"[hyper, ##para, ##met, ##rization]","[101, 23760, 28689, 11368, 26910, 102]",hyperparametrization
6,distillation,"[di, ##sti, ##llation]","[101, 4487, 16643, 20382, 102]",distillation
7,contextualization,"[context, ##ual, ##ization]","[101, 6123, 8787, 3989, 102]",contextualization


### Conclusión del apartado 1.a

WordPiece permite representar palabras desconocidas dividiéndolas en unidades más pequeñas.

El prefijo `##` indica que el subtoken es una continuación de la palabra anterior.

Esto reduce el problema de palabras fuera de vocabulario, porque el modelo puede construir palabras complejas a partir de subtokens conocidos.


## Ejercicio 1.b — Tokens especiales y máscara de atención

BERT no solo convierte palabras en ids. También añade tokens especiales:

- `[CLS]`: *ClasSification*, inicio de secuencia. En clasificación se usa como representación global.
- `[SEP]`: *SEParator*, final de secuencia o separación entre dos frases.
- `[PAD]`: *PADding*, relleno cuando hace falta igualar longitudes.

La `attention_mask` indica qué tokens son reales y cuáles son padding.


In [7]:
texts = [
    "Transformers changed NLP.",
    "Tokenizers convert text into ids.",
    "Padding and truncation are practical decisions.",
    "Batch length affects computational cost.",
]

texts

['Transformers changed NLP.',
 'Tokenizers convert text into ids.',
 'Padding and truncation are practical decisions.',
 'Batch length affects computational cost.']

### 1.b.1 Tokenizar con `add_special_tokens=True`

Usamos:

```python
add_special_tokens=True
```

Después convertimos ids a tokens para ver exactamente qué ha añadido el tokenizador.


In [8]:
for text in texts:
    encoding = bert_tokenizer(
        text,
        add_special_tokens=True,
        return_attention_mask=True
    )

    input_ids = encoding["input_ids"]
    tokens = bert_tokenizer.convert_ids_to_tokens(input_ids)
    attention_mask = encoding["attention_mask"]

    print("Texto:", text)
    print("Input IDs:", input_ids)
    print("Tokens:", tokens)
    print("Attention mask:", attention_mask)
    print("-" * 100)

Texto: Transformers changed NLP.
Input IDs: [101, 19081, 2904, 17953, 2361, 1012, 102]
Tokens: ['[CLS]', 'transformers', 'changed', 'nl', '##p', '.', '[SEP]']
Attention mask: [1, 1, 1, 1, 1, 1, 1]
----------------------------------------------------------------------------------------------------
Texto: Tokenizers convert text into ids.
Input IDs: [101, 19204, 17629, 2015, 10463, 3793, 2046, 8909, 2015, 1012, 102]
Tokens: ['[CLS]', 'token', '##izer', '##s', 'convert', 'text', 'into', 'id', '##s', '.', '[SEP]']
Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
----------------------------------------------------------------------------------------------------
Texto: Padding and truncation are practical decisions.
Input IDs: [101, 11687, 4667, 1998, 19817, 4609, 10719, 2024, 6742, 6567, 1012, 102]
Tokens: ['[CLS]', 'pad', '##ding', 'and', 'tr', '##un', '##cation', 'are', 'practical', 'decisions', '.', '[SEP]']
Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
-------------------------

### 1.b.2 Tabla compacta

Guardamos la información en una tabla para analizarlo más cómodamente.


In [9]:
rows = []

for text in texts:
    encoding = bert_tokenizer(
        text,
        add_special_tokens=True,
        return_attention_mask=True
    )

    input_ids = encoding["input_ids"]
    tokens = bert_tokenizer.convert_ids_to_tokens(input_ids)
    attention_mask = encoding["attention_mask"]

    rows.append({
        "Texto": text,
        "Input IDs": input_ids,
        "Tokens": tokens,
        "Attention mask": attention_mask
    })

df_special_tokens = pd.DataFrame(rows)
df_special_tokens

,Texto,Input IDs,Tokens,Attention mask
0,Transformers changed NLP.,"[101, 19081, 2904, 17953, 2361, 1012, 102]","[[CLS], transformers, changed, nl, ##p, ., [SEP]]","[1, 1, 1, 1, 1, 1, 1]"
1,Tokenizers convert text into ids.,"[101, 19204, 17629, 2015, 10463, 3793, 2046, 8...","[[CLS], token, ##izer, ##s, convert, text, int...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
2,Padding and truncation are practical decisions.,"[101, 11687, 4667, 1998, 19817, 4609, 10719, 2...","[[CLS], pad, ##ding, and, tr, ##un, ##cation, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
3,Batch length affects computational cost.,"[101, 14108, 3091, 13531, 15078, 3465, 1012, 102]","[[CLS], batch, length, affects, computational,...","[1, 1, 1, 1, 1, 1, 1, 1]"


### Conclusión del apartado 1.b

- `[CLS]` aparece al inicio de la secuencia.
- `[SEP]` aparece al final.
- Cuando no hay padding, la máscara de atención está formada por unos.
- La máscara vale `1` en tokens que el modelo debe atender.
- Si apareciera padding, la máscara valdría `0` en esas posiciones.


## Ejercicio 1.c — Padding, truncado y longitud máxima

En la práctica, las secuencias no tienen todas la misma longitud.

Por eso usamos:

- **Truncado**: recortar secuencias demasiado largas.
- **Padding**: rellenar secuencias demasiado cortas.

BERT admite hasta 512 tokens, pero en este ejercicio usaremos longitudes más pequeñas para ver claramente el efecto.


### 1.c.1 Truncado con `max_length=4`

Tokenizamos con:

```python
truncation=True
max_length=4
```

No fijamos longitud de relleno.

Como BERT añade `[CLS]` y `[SEP]`, solo quedan dos posiciones útiles para contenido real.


In [10]:
for text in texts:
    encoding = bert_tokenizer(
        text,
        add_special_tokens=True,
        truncation=True,
        max_length=4
    )

    input_ids = encoding["input_ids"]
    tokens = bert_tokenizer.convert_ids_to_tokens(input_ids)

    print("Texto original:", text)
    print("Tokens conservados:", tokens)
    print("Longitud:", len(tokens))
    print("-" * 100)

Texto original: Transformers changed NLP.
Tokens conservados: ['[CLS]', 'transformers', 'changed', '[SEP]']
Longitud: 4
----------------------------------------------------------------------------------------------------
Texto original: Tokenizers convert text into ids.
Tokens conservados: ['[CLS]', 'token', '##izer', '[SEP]']
Longitud: 4
----------------------------------------------------------------------------------------------------
Texto original: Padding and truncation are practical decisions.
Tokens conservados: ['[CLS]', 'pad', '##ding', '[SEP]']
Longitud: 4
----------------------------------------------------------------------------------------------------
Texto original: Batch length affects computational cost.
Tokens conservados: ['[CLS]', 'batch', 'length', '[SEP]']
Longitud: 4
----------------------------------------------------------------------------------------------------


### Análisis

Con `max_length=4` se pierde mucha información.

Ejemplo conceptual:

```text
[CLS] token1 token2 [SEP]
```

Solo hay dos posiciones para palabras reales. Por tanto, el truncado excesivo puede eliminar partes importantes de la frase.


### 1.c.2 Comparar `padding=True` y `padding="max_length"`

`padding=True` rellena hasta la secuencia más larga del batch actual.

`padding="max_length"` rellena siempre hasta la longitud fija indicada por `max_length`.


In [13]:
encoding_padding_true = bert_tokenizer(
    texts,
    add_special_tokens=True,
    padding=True,
    truncation=True,
    return_attention_mask=True
)

encoding_padding_max_length = bert_tokenizer(
    texts,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=12,
    return_attention_mask=True
)

print("=== padding=True ===")
for i, text in enumerate(texts):
    ids = encoding_padding_true["input_ids"][i]
    tokens = bert_tokenizer.convert_ids_to_tokens(ids)
    mask = encoding_padding_true["attention_mask"][i]

    print("Texto:", text)
    print("Longitud:", len(ids))
    print("Tokens:", tokens)
    print("Attention mask:", mask)
    print("-" * 100)

print("\n=== padding='max_length', max_length=12 ===")
for i, text in enumerate(texts):
    ids = encoding_padding_max_length["input_ids"][i]
    tokens = bert_tokenizer.convert_ids_to_tokens(ids)
    mask = encoding_padding_max_length["attention_mask"][i]

    print("Texto:", text)
    print("Longitud:", len(ids))
    print("Tokens:", tokens)
    print("Attention mask:", mask)
    print("-" * 100)

=== padding=True ===
Texto: Transformers changed NLP.
Longitud: 12
Tokens: ['[CLS]', 'transformers', 'changed', 'nl', '##p', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
Attention mask: [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0]
----------------------------------------------------------------------------------------------------
Texto: Tokenizers convert text into ids.
Longitud: 12
Tokens: ['[CLS]', 'token', '##izer', '##s', 'convert', 'text', 'into', 'id', '##s', '.', '[SEP]', '[PAD]']
Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0]
----------------------------------------------------------------------------------------------------
Texto: Padding and truncation are practical decisions.
Longitud: 12
Tokens: ['[CLS]', 'pad', '##ding', 'and', 'tr', '##un', '##cation', 'are', 'practical', 'decisions', '.', '[SEP]']
Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
----------------------------------------------------------------------------------------------------
Texto: Bat

### 1.c.3 Tabla comparativa de longitudes

Comparamos la longitud obtenida en cada secuencia.


In [14]:
rows = []

for i, text in enumerate(texts):
    rows.append({
        "Texto": text,
        "Longitud con padding=True": len(encoding_padding_true["input_ids"][i]),
        "Longitud con padding='max_length' (12)": len(encoding_padding_max_length["input_ids"][i])
    })

df_padding_comparison = pd.DataFrame(rows)
df_padding_comparison

,Texto,Longitud con padding=True,Longitud con padding='max_length' (12)
0,Transformers changed NLP.,12,12
1,Tokenizers convert text into ids.,12,12
2,Padding and truncation are practical decisions.,12,12
3,Batch length affects computational cost.,12,12


### Conclusión sobre padding

- `padding=True` depende del batch actual.
- `padding="max_length"` fuerza una longitud fija.

`padding=True` puede ser más eficiente porque evita rellenar de más.  
`padding="max_length"` es más rígido, pero garantiza entradas de longitud constante.


### 1.c.4 Tokenizar con distintos valores de `max_length`

Ahora usamos:

```python
padding="max_length"
truncation=True
```

con varios valores de `max_length`.

Mostraremos:

- Tokens conservados.
- Máscara de atención.
- Posiciones donde aparece `[PAD]`.


In [16]:
max_lengths = [4, 8, 12, 16]

for max_len in max_lengths:
    print("=" * 120)
    print(f"MAX_LENGTH = {max_len}")
    print("=" * 120)

    encodings = bert_tokenizer(
        texts,
        add_special_tokens=True,
        padding="max_length",
        truncation=True,
        max_length=max_len,
        return_attention_mask=True
    )

    for i, text in enumerate(texts):
        ids = encodings["input_ids"][i]
        tokens = bert_tokenizer.convert_ids_to_tokens(ids)
        mask = encodings["attention_mask"][i]

        pad_positions = [
            pos for pos, token in enumerate(tokens)
            if token == "[PAD]"
        ]

        print("Texto:", text)
        print("Longitud final:", len(ids))
        print("Tokens:", tokens)
        print("Attention mask:", mask)
        print("Posiciones con [PAD]:", pad_positions)
        print("-" * 100)

MAX_LENGTH = 4
Texto: Transformers changed NLP.
Longitud final: 4
Tokens: ['[CLS]', 'transformers', 'changed', '[SEP]']
Attention mask: [1, 1, 1, 1]
Posiciones con [PAD]: []
----------------------------------------------------------------------------------------------------
Texto: Tokenizers convert text into ids.
Longitud final: 4
Tokens: ['[CLS]', 'token', '##izer', '[SEP]']
Attention mask: [1, 1, 1, 1]
Posiciones con [PAD]: []
----------------------------------------------------------------------------------------------------
Texto: Padding and truncation are practical decisions.
Longitud final: 4
Tokens: ['[CLS]', 'pad', '##ding', '[SEP]']
Attention mask: [1, 1, 1, 1]
Posiciones con [PAD]: []
----------------------------------------------------------------------------------------------------
Texto: Batch length affects computational cost.
Longitud final: 4
Tokens: ['[CLS]', 'batch', 'length', '[SEP]']
Attention mask: [1, 1, 1, 1]
Posiciones con [PAD]: []
---------------------------

### 1.c.5 Tabla resumen con distintos `max_length`

Mostramos una tabla con:

- Longitud final.
- Número de tokens reales.
- Número de tokens `[PAD]`.


In [17]:
rows = []

for max_len in max_lengths:
    encodings = bert_tokenizer(
        texts,
        add_special_tokens=True,
        padding="max_length",
        truncation=True,
        max_length=max_len,
        return_attention_mask=True
    )

    for i, text in enumerate(texts):
        ids = encodings["input_ids"][i]
        tokens = bert_tokenizer.convert_ids_to_tokens(ids)
        mask = encodings["attention_mask"][i]

        rows.append({
            "max_length": max_len,
            "Texto": text,
            "Longitud final": len(ids),
            "Tokens reales según attention_mask": sum(mask),
            "Nº de [PAD]": tokens.count("[PAD]"),
            "Tokens": tokens,
            "Attention mask": mask
        })

df_max_length = pd.DataFrame(rows)
df_max_length

,max_length,Texto,Longitud final,Tokens reales según attention_mask,Nº de [PAD],Tokens,Attention mask
0,4,Transformers changed NLP.,4,4,0,"[[CLS], transformers, changed, [SEP]]","[1, 1, 1, 1]"
1,4,Tokenizers convert text into ids.,4,4,0,"[[CLS], token, ##izer, [SEP]]","[1, 1, 1, 1]"
2,4,Padding and truncation are practical decisions.,4,4,0,"[[CLS], pad, ##ding, [SEP]]","[1, 1, 1, 1]"
3,4,Batch length affects computational cost.,4,4,0,"[[CLS], batch, length, [SEP]]","[1, 1, 1, 1]"
4,8,Transformers changed NLP.,8,7,1,"[[CLS], transformers, changed, nl, ##p, ., [SE...","[1, 1, 1, 1, 1, 1, 1, 0]"
5,8,Tokenizers convert text into ids.,8,8,0,"[[CLS], token, ##izer, ##s, convert, text, int...","[1, 1, 1, 1, 1, 1, 1, 1]"
6,8,Padding and truncation are practical decisions.,8,8,0,"[[CLS], pad, ##ding, and, tr, ##un, ##cation, ...","[1, 1, 1, 1, 1, 1, 1, 1]"
7,8,Batch length affects computational cost.,8,8,0,"[[CLS], batch, length, affects, computational,...","[1, 1, 1, 1, 1, 1, 1, 1]"
8,12,Transformers changed NLP.,12,7,5,"[[CLS], transformers, changed, nl, ##p, ., [SE...","[1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0]"
9,12,Tokenizers convert text into ids.,12,11,1,"[[CLS], token, ##izer, ##s, convert, text, int...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0]"


### Conclusión del apartado 1.c

Existe un compromiso entre coste computacional y contenido preservado.

- Si `max_length` es demasiado pequeño, se pierde información por truncamiento.
- Si `max_length` es demasiado grande, aparecen muchos `[PAD]`.
- Aunque la `attention_mask` permite ignorar `[PAD]`, procesar secuencias largas sigue teniendo más coste.

Por tanto, hay que elegir un valor de `max_length` equilibrado.


# Ejercicio 2. Tokenizadores específicos de dominio

Ahora comparamos:

- `bert-base-uncased`: tokenizador generalista.
- `microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext`: tokenizador biomédico.

La idea es comprobar si un tokenizador especializado fragmenta menos los términos técnicos biomédicos.


In [18]:
biomedical_texts = [
    "Immunohistochemistry confirmed hepatocellular adenocarcinoma with extensive "
    "lymphovascular invasion. Pharmacokinetic analysis of the oligonucleotide "
    "therapy revealed phosphorylation-dependent degradation by endonucleases. "
    "Chemotherapy with paclitaxel and trastuzumab induced angiogenesis inhibition "
    "and downregulated glycoprotein expression in metastatic lymphocytes."
]

biomedical_texts[0]

'Immunohistochemistry confirmed hepatocellular adenocarcinoma with extensive lymphovascular invasion. Pharmacokinetic analysis of the oligonucleotide therapy revealed phosphorylation-dependent degradation by endonucleases. Chemotherapy with paclitaxel and trastuzumab induced angiogenesis inhibition and downregulated glycoprotein expression in metastatic lymphocytes.'

## 2.1 Tokenización con BERT generalista

Primero tokenizamos el texto biomédico con `bert-base-uncased`.

Al ser un tokenizador generalista, es esperable que fragmente muchos términos técnicos.


In [19]:
bert_biomedical_tokens = bert_tokenizer.tokenize(biomedical_texts[0])

print("Número de tokens con bert-base-uncased:", len(bert_biomedical_tokens))
print("\nTokens:")
print(bert_biomedical_tokens)

Número de tokens con bert-base-uncased: 92

Tokens:
['im', '##mun', '##oh', '##isto', '##chemist', '##ry', 'confirmed', 'he', '##pa', '##to', '##cellular', 'aden', '##oca', '##rc', '##ino', '##ma', 'with', 'extensive', 'l', '##ym', '##ph', '##ova', '##scu', '##lar', 'invasion', '.', 'ph', '##arm', '##aco', '##kin', '##etic', 'analysis', 'of', 'the', 'ol', '##igo', '##nu', '##cle', '##otide', 'therapy', 'revealed', 'ph', '##os', '##ph', '##ory', '##lation', '-', 'dependent', 'degradation', 'by', 'end', '##on', '##uc', '##lea', '##ses', '.', 'chemotherapy', 'with', 'pac', '##lita', '##x', '##el', 'and', 'tr', '##ast', '##uz', '##uma', '##b', 'induced', 'ang', '##io', '##genesis', 'inhibition', 'and', 'down', '##re', '##gul', '##ated', 'g', '##ly', '##co', '##pro', '##tein', 'expression', 'in', 'meta', '##static', 'l', '##ym', '##ph', '##ocytes', '.']


## 2.2 Tokenización con PubMedBERT

Ahora cargamos el tokenizador de PubMedBERT.

Este tokenizador está entrenado sobre textos biomédicos, por lo que debería adaptarse mejor a términos clínicos y científicos.


In [20]:
pubmedbert_tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"
)

pubmedbert_tokens = pubmedbert_tokenizer.tokenize(biomedical_texts[0])

print("Tokenizador cargado:", pubmedbert_tokenizer.name_or_path)
print("Número de tokens con PubMedBERT:", len(pubmedbert_tokens))
print("\nTokens:")
print(pubmedbert_tokens)

Tokenizador cargado: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Número de tokens con PubMedBERT: 41

Tokens:
['immunohistochemistry', 'confirmed', 'hepatocellular', 'adenocarcinoma', 'with', 'extensive', 'lymph', '##ovascular', 'invasion', '.', 'pharmacokinetic', 'analysis', 'of', 'the', 'oligonucleotide', 'therapy', 'revealed', 'phosphorylation', '-', 'dependent', 'degradation', 'by', 'endonuclease', '##s', '.', 'chemotherapy', 'with', 'paclitaxel', 'and', 'trastuzumab', 'induced', 'angiogenesis', 'inhibition', 'and', 'downregulated', 'glycoprotein', 'expression', 'in', 'metastatic', 'lymphocytes', '.']


## 2.3 Comparación directa

Creamos una tabla con el número total de tokens generado por cada tokenizador.

Si un tokenizador genera menos tokens para el mismo texto técnico, suele indicar que fragmenta menos las palabras especializadas.


In [21]:
comparison = pd.DataFrame({
    "Tokenizador": [
        "bert-base-uncased",
        "PubMedBERT"
    ],
    "Número de tokens": [
        len(bert_biomedical_tokens),
        len(pubmedbert_tokens)
    ],
    "Tokens": [
        bert_biomedical_tokens,
        pubmedbert_tokens
    ]
})

comparison

,Tokenizador,Número de tokens,Tokens
0,bert-base-uncased,92,"[im, ##mun, ##oh, ##isto, ##chemist, ##ry, con..."
1,PubMedBERT,41,"[immunohistochemistry, confirmed, hepatocellul..."


## 2.4 Comparación sobre términos biomédicos concretos

Para ver mejor la diferencia, comparamos la tokenización de términos técnicos aislados.


In [22]:
technical_terms = [
    "Immunohistochemistry",
    "hepatocellular",
    "adenocarcinoma",
    "lymphovascular",
    "Pharmacokinetic",
    "oligonucleotide",
    "phosphorylation",
    "endonucleases",
    "paclitaxel",
    "trastuzumab",
    "angiogenesis",
    "glycoprotein",
    "lymphocytes"
]

rows = []

for term in technical_terms:
    bert_tokens = bert_tokenizer.tokenize(term)
    pubmed_tokens = pubmedbert_tokenizer.tokenize(term)

    rows.append({
        "Término": term,
        "BERT tokens": bert_tokens,
        "Nº tokens BERT": len(bert_tokens),
        "PubMedBERT tokens": pubmed_tokens,
        "Nº tokens PubMedBERT": len(pubmed_tokens)
    })

df_terms = pd.DataFrame(rows)
df_terms

,Término,BERT tokens,Nº tokens BERT,PubMedBERT tokens,Nº tokens PubMedBERT
0,Immunohistochemistry,"[im, ##mun, ##oh, ##isto, ##chemist, ##ry]",6,[immunohistochemistry],1
1,hepatocellular,"[he, ##pa, ##to, ##cellular]",4,[hepatocellular],1
2,adenocarcinoma,"[aden, ##oca, ##rc, ##ino, ##ma]",5,[adenocarcinoma],1
3,lymphovascular,"[l, ##ym, ##ph, ##ova, ##scu, ##lar]",6,"[lymph, ##ovascular]",2
4,Pharmacokinetic,"[ph, ##arm, ##aco, ##kin, ##etic]",5,[pharmacokinetic],1
5,oligonucleotide,"[ol, ##igo, ##nu, ##cle, ##otide]",5,[oligonucleotide],1
6,phosphorylation,"[ph, ##os, ##ph, ##ory, ##lation]",5,[phosphorylation],1
7,endonucleases,"[end, ##on, ##uc, ##lea, ##ses]",5,"[endonuclease, ##s]",2
8,paclitaxel,"[pac, ##lita, ##x, ##el]",4,[paclitaxel],1
9,trastuzumab,"[tr, ##ast, ##uz, ##uma, ##b]",5,[trastuzumab],1


## 2.5 Análisis de fragmentación

Una palabra está más fragmentada si se divide en más subtokens.

Comparamos solo el número de tokens que genera cada tokenizador para cada término técnico.


In [ ]:
df_terms[[
    "Término",
    "Nº tokens BERT",
    "Nº tokens PubMedBERT"
]]

## Conclusión del Ejercicio 2

El tokenizador de `bert-base-uncased` es generalista. Puede funcionar bien para inglés común, pero en textos biomédicos tiende a fragmentar términos técnicos.

PubMedBERT está especializado en textos biomédicos. Por eso puede representar mejor términos como nombres de tratamientos, procesos moleculares o enfermedades.

Esto puede mejorar tareas de PLN biomédico porque el modelo trabaja con unidades más significativas para el dominio.

Resumen:

```text
Texto general en inglés → BERT generalista
Texto biomédico → PubMedBERT u otro modelo especializado
```


# Conclusión general de la hoja

En esta hoja hemos visto que la tokenización es una fase fundamental en los modelos Transformer.

Ideas clave:

1. WordPiece divide palabras poco frecuentes en subtokens.
2. El prefijo `##` indica continuación de palabra.
3. `[CLS]` marca el inicio y suele usarse en clasificación.
4. `[SEP]` marca el final de la secuencia.
5. `[PAD]` rellena secuencias cortas.
6. `attention_mask` indica qué posiciones debe atender el modelo.
7. `truncation=True` evita superar la longitud máxima, pero puede eliminar información.
8. `padding=True` depende del batch actual.
9. `padding="max_length"` fuerza una longitud fija.
10. Los tokenizadores especializados pueden mejorar la representación de textos técnicos.

La elección del tokenizador puede cambiar mucho cómo el modelo interpreta el texto.
